# UIUC Campus LiDAR — Detection & Segmentation (reproducible notebook)

This notebook is **fully self-contained**: it downloads the dataset from I-GUIDE,
merges the tiles into one gap-free cloud, and runs both a **classical** object-detection
pipeline (ground/DTM, building instances, individual trees) and a **deep-learning**
semantic-segmentation model (PointNet, trained on the ASPRS labels).

**Source data:** USGS 3DEP LPC, `IL_8County_PlusChampaign_2019_B19` (QL1), 4 × 1 km tiles.
**CRS:** NAD83(2011) / Conus Albers metres (EPSG:6350), vertical NAVD88 (Geoid12B).

Run top-to-bottom. Data downloads once; heavy passes are cached in memory across cells.

## 0 · Install dependencies

In [ ]:
import sys, subprocess
pkgs = ["laspy[lazrs]", "scikit-image", "shapely", "rasterio",
        "pyproj", "scipy", "scikit-learn", "matplotlib", "torch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
print("dependencies ready")

## 1 · Imports & configuration

In [ ]:
%matplotlib inline
import os, glob, json, zipfile, numpy as np, laspy
import matplotlib.pyplot as plt
from urllib.request import urlretrieve

WORK = "uiuc_campus_lidar"                 # working / output folder
OUT  = os.path.join(WORK, "results")
os.makedirs(OUT, exist_ok=True)
RES  = 0.5                                 # raster resolution (m)
DATASET_URL = "https://storage.i-guide.io/datasets/9c1842f8-1d74-4b88-8ad3-cf0a0cb47a86/uiuc_campus_lidar.zip"
print("workdir:", os.path.abspath(WORK))

## 2 · Download the dataset from I-GUIDE
`urlretrieve` fetches the ZIP (skipped if already present), then we extract and locate
the point-cloud file(s).

In [ ]:
zip_path = os.path.join(WORK, "uiuc_campus_lidar.zip")
if not os.path.exists(zip_path):
    print("downloading dataset ...")
    urlretrieve(DATASET_URL, zip_path)
print("zip:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")

with zipfile.ZipFile(zip_path) as z:
    z.extractall(WORK)

lazs = sorted(glob.glob(os.path.join(WORK, "**", "*.laz"), recursive=True) +
              glob.glob(os.path.join(WORK, "**", "*.las"), recursive=True))
print(f"found {len(lazs)} point-cloud file(s):")
for p in lazs: print("  ", os.path.relpath(p, WORK))

## 3 · Merge tiles into one gap-free cloud
If the archive already contains a merged file we use it; otherwise the adjacent tiles are
concatenated (they share edges exactly, so this is lossless — no interpolation).

In [ ]:
def merge_tiles(files, out):
    with laspy.open(files[0]) as r0:
        header = r0.header
    total = 0
    with laspy.open(out, mode="w", header=header) as w:
        for f in files:
            with laspy.open(f) as r:
                for pts in r.chunk_iterator(5_000_000):
                    w.write_points(pts); total += len(pts)
    print(f"merged {len(files)} tiles -> {out}  ({total:,} pts)")
    return out

merged = [p for p in lazs if "merged" in os.path.basename(p).lower()]
if merged:
    SRC = merged[0]
elif len(lazs) == 1:
    SRC = lazs[0]
else:
    SRC = merge_tiles(lazs, os.path.join(WORK, "UIUC_campus_LiDAR_merged_2x2km.laz"))

h = laspy.open(SRC).header
xmin, ymin, xmax, ymax = h.mins[0], h.mins[1], h.maxs[0], h.maxs[1]
nx = int(round((xmax - xmin) / RES)); ny = int(round((ymax - ymin) / RES))
def rc(x, y):
    col = np.clip(((x - xmin)/RES).astype(np.int64), 0, nx-1)
    row = np.clip(((ymax - y)/RES).astype(np.int64), 0, ny-1)
    return row*nx + col
print(f"SRC = {SRC}\npoints = {h.point_count:,}   extent {xmax-xmin:.0f} x {ymax-ymin:.0f} m")

## 4 · Single rasterization pass  →  surface, terrain, class counts, coverage
One streaming pass folds all 80 M points into 0.5 m grids and tallies the ASPRS classes.
Noise classes (7, 18) are excluded from the surface model.

In [ ]:
ASPRS = {0:"Created",1:"Unclassified",2:"Ground",3:"Low Veg",4:"Med Veg",5:"High Veg",
         6:"Building",7:"Low Point (noise)",9:"Water",10:"Rail",11:"Road Surface",
         17:"Bridge Deck",18:"High Noise"}
dsm      = np.full(ny*nx, -np.inf, np.float32)   # max Z (surface, noise removed)
grd      = np.full(ny*nx,  np.inf, np.float32)   # min Z of ground (class 2)
bldg_cnt = np.zeros(ny*nx, np.int32)             # class 6 count
veg_cnt  = np.zeros(ny*nx, np.int32)             # class 5 count
class_hist = np.zeros(256, np.int64)

with laspy.open(SRC) as r:
    for pts in r.chunk_iterator(10_000_000):
        x, y, z = np.asarray(pts.x), np.asarray(pts.y), np.asarray(pts.z)
        c = np.asarray(pts.classification); idx = rc(x, y)
        class_hist += np.bincount(c, minlength=256)
        surf = (c != 7) & (c != 18)
        np.maximum.at(dsm, idx[surf], z[surf])
        g = c == 2
        if g.any(): np.minimum.at(grd, idx[g], z[g])
        b = c == 6
        if b.any(): np.add.at(bldg_cnt, idx[b], 1)
        v = c == 5
        if v.any(): np.add.at(veg_cnt, idx[v], 1)

dsm = dsm.reshape(ny,nx); grd = grd.reshape(ny,nx)
bldg_cnt = bldg_cnt.reshape(ny,nx); veg_cnt = veg_cnt.reshape(ny,nx)
dsm[~np.isfinite(dsm)] = np.nan; grd[~np.isfinite(grd)] = np.nan

# coverage check
empty = int(np.isnan(dsm).sum())
print(f"coverage: {100*(1-empty/(ny*nx)):.3f}% of cells populated "
      f"({empty:,} empty = water / specular roofs)")

from matplotlib.colors import LightSource
ls = LightSource(azdeg=315, altdeg=45)
filled = np.where(np.isnan(dsm), np.nanmin(dsm), dsm)
rgb = ls.shade(filled, cmap=plt.cm.terrain,
               vmin=np.nanpercentile(dsm,1), vmax=np.nanpercentile(dsm,99),
               blend_mode="soft", vert_exag=2.0)
rgb[np.isnan(dsm)] = [1,0,1,1]
plt.figure(figsize=(9,9)); plt.imshow(rgb, extent=[xmin,xmax,ymin,ymax], origin="upper")
plt.title("Coverage check — 1 m DSM hillshade (magenta = no data)")
plt.ticklabel_format(style="plain"); plt.tight_layout(); plt.show()

## 5 · Georeference & encoding metadata (for GIS / OSM hand-off)

In [ ]:
from pyproj import Transformer
from shapely.geometry import Polygon, mapping
from shapely.ops import transform as shp_transform
to_ll = Transformer.from_crs("EPSG:6350", "EPSG:4326", always_xy=True)

# bounding box in lon/lat (densify edges; Albers grid is rotated vs lon/lat)
n = 40
ring = ([(x,ymax) for x in np.linspace(xmin,xmax,n)] +
        [(xmax,y) for y in np.linspace(ymax,ymin,n)] +
        [(x,ymin) for x in np.linspace(xmax,xmin,n)] +
        [(xmin,y) for y in np.linspace(ymin,ymax,n)])
ll = [to_ll.transform(x,y) for x,y in ring]
lons=[p[0] for p in ll]; lats=[p[1] for p in ll]
print("Horizontal CRS : EPSG:6350 (NAD83(2011) Conus Albers, m)")
print("Vertical CRS   : EPSG:5703 (NAVD88 height, Geoid12B)")
print(f"WGS84 bbox     : {min(lons):.6f},{min(lats):.6f},{max(lons):.6f},{max(lats):.6f}")
poly_ll = shp_transform(lambda xs,ys,z=None: to_ll.transform(xs,ys),
                        Polygon([(xmin,ymin),(xmax,ymin),(xmax,ymax),(xmin,ymax)]))
json.dump({"type":"FeatureCollection","features":[{"type":"Feature",
           "properties":{"name":"UIUC campus LiDAR footprint"},
           "geometry":mapping(poly_ll)}]},
          open(os.path.join(OUT,"footprint.geojson"),"w"))

print("\nASPRS classification (encoding):")
tot = class_hist.sum()
for c in np.nonzero(class_hist)[0]:
    print(f"  {c:>3}  {ASPRS.get(int(c),'User/Reserved'):<20} {class_hist[c]:>12,}  {100*class_hist[c]/tot:5.2f}%")

## 6 · Ground / DTM surface
Fill terrain no-data (under buildings, water) by nearest ground value, then derive the
Canopy Height Model `CHM = DSM − DTM` — the key height-above-ground feature reused below.

In [ ]:
from scipy import ndimage as ndi
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
transform = from_origin(xmin, ymax, RES, RES); albers = CRS.from_epsg(6350)

_, (ir, ic) = ndi.distance_transform_edt(np.isnan(grd), return_indices=True)
dtm = grd[ir, ic].astype(np.float32)
chm = np.clip(np.where(np.isnan(dsm),0,dsm) - dtm, 0, None).astype(np.float32)
for name, arr, nd in [("dtm",dtm,np.nan), ("chm",chm,0)]:
    with rasterio.open(os.path.join(OUT,f"{name}.tif"),"w",driver="GTiff",height=ny,width=nx,
                       count=1,dtype="float32",crs=albers,transform=transform,nodata=nd) as d:
        d.write(arr,1)

plt.figure(figsize=(8,8))
plt.imshow(ls.hillshade(dtm,vert_exag=3,dx=RES,dy=RES),cmap="gray",extent=[xmin,xmax,ymin,ymax])
plt.imshow(dtm,cmap="terrain",alpha=.5,extent=[xmin,xmax,ymin,ymax])
plt.colorbar(shrink=.7,label="NAVD88 (m)"); plt.title("Bare-earth DTM")
plt.ticklabel_format(style="plain"); plt.tight_layout(); plt.show()

## 7 · Building instance detection
Rasterize class-6 points → clean with morphology → connected components. Each component is
one building; height comes from the CHM. Footprints are exported to WGS84 GeoJSON.

In [ ]:
from skimage.morphology import remove_small_objects, closing, disk, remove_small_holes
from skimage.measure import regionprops
from rasterio import features
from shapely.geometry import shape

min_cells = int(20.0/(RES*RES))                      # 20 m^2 minimum footprint
bmask = closing(bldg_cnt >= 2, disk(2))
bmask = remove_small_holes(bmask, area_threshold=min_cells)
bmask = remove_small_objects(bmask, min_size=min_cells)
lbl, _ = ndi.label(bmask)
keep = {p.label for p in regionprops(lbl) if np.nanmedian(chm[lbl==p.label]) >= 2.0}
bmask = np.isin(lbl, list(keep)); lbl2, nb = ndi.label(bmask)

feats, bb = [], []
for p in regionprops(lbl2):
    hgt = float(np.nanmedian(chm[lbl2==p.label])); area = float(p.area*RES*RES)
    minr,minc,maxr,maxc = p.bbox
    bb.append((xmin+minc*RES, ymax-maxr*RES, xmin+maxc*RES, ymax-minr*RES))
    feats.append(dict(label=int(p.label), area_m2=round(area,1), height_m=round(hgt,1)))
geoms=[]
for geom,val in features.shapes(lbl2.astype(np.int32),mask=bmask,transform=transform):
    if val==0: continue
    poly = shp_transform(lambda xs,ys,z=None: to_ll.transform(xs,ys), shape(geom).simplify(0.5))
    m = next(f for f in feats if f["label"]==int(val))
    geoms.append(dict(type="Feature",properties={**m,"class":"building"},geometry=mapping(poly)))
json.dump(dict(type="FeatureCollection",features=geoms), open(os.path.join(OUT,"buildings.geojson"),"w"))
print(f"{nb} buildings | footprint {sum(f['area_m2'] for f in feats)/1e6:.2f} km^2 | "
      f"median height {np.median([f['height_m'] for f in feats]):.1f} m")

rng = np.random.default_rng(0)
inst = np.vstack([[0,0,0], rng.uniform(.25,1,(nb,3))])[lbl2]
plt.figure(figsize=(10,10))
plt.imshow(ls.hillshade(np.nan_to_num(dsm,nan=np.nanmin(dsm)),vert_exag=2,dx=RES,dy=RES),
           cmap="gray",extent=[xmin,xmax,ymin,ymax])
plt.imshow(np.dstack([inst,(lbl2>0)*.55]),extent=[xmin,xmax,ymin,ymax])
ax=plt.gca()
for x0,y0,x1,y1 in bb: ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,ec="red",lw=.5))
plt.title(f"Building instances — {nb}"); plt.xlim(xmin,xmax); plt.ylim(ymin,ymax)
plt.ticklabel_format(style="plain"); plt.tight_layout(); plt.show()

## 8 · Individual tree detection
Local maxima on the smoothed CHM (≥3 m, ≥3 m spacing) give tree tops; watershed on the
inverted CHM delineates crowns. Buildings are masked out first.

In [ ]:
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.filters import gaussian

region = (chm > 3.0) & (veg_cnt >= 1) & (~ndi.binary_dilation(bmask, disk(2)))
chm_s = gaussian(chm, sigma=1.0, preserve_range=True); chm_s[~region] = 0
tops = peak_local_max(chm_s, min_distance=6, threshold_abs=3.0, labels=region)
markers = np.zeros_like(chm_s, np.int32)
for i,(r_,c_) in enumerate(tops,1): markers[r_,c_]=i
crowns = watershed(-chm_s, markers, mask=region); nt=len(tops)
area = ndi.sum(np.ones_like(crowns), crowns, index=np.arange(1,nt+1))*RES*RES
tf=[]
for i,(r_,c_) in enumerate(tops):
    lon,lat = to_ll.transform(xmin+c_*RES, ymax-r_*RES)
    tf.append(dict(type="Feature",properties=dict(tree_id=i+1,height_m=round(float(chm[r_,c_]),1),
              crown_m2=round(float(area[i]),1),**{"class":"tree"}),
              geometry=dict(type="Point",coordinates=[round(lon,8),round(lat,8)])))
json.dump(dict(type="FeatureCollection",features=tf), open(os.path.join(OUT,"trees.geojson"),"w"))
print(f"{nt} trees | median height {np.median([f['properties']['height_m'] for f in tf]):.1f} m")

plt.figure(figsize=(10,10))
plt.imshow(np.where(region,chm,np.nan),cmap="YlGn",vmax=25,extent=[xmin,xmax,ymin,ymax])
plt.scatter(xmin+tops[:,1]*RES, ymax-tops[:,0]*RES, s=2, c="darkred", marker="^")
plt.colorbar(shrink=.7,label="canopy height (m)"); plt.title(f"Individual trees — {nt}")
plt.xlim(xmin,xmax); plt.ylim(ymin,ymax); plt.ticklabel_format(style="plain")
plt.tight_layout(); plt.show()

## 9 · Deep-learning semantic segmentation (PointNet)
Predict the 5 ASPRS semantic classes from geometry + features. Ground truth = the
classification in the LAZ. **Spatial split** (west train / east val) prevents leakage.
Trains on CUDA → MPS → CPU automatically.

In [ ]:
import torch, torch.nn as nn
dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
BLOCK, NPTS = 40.0, 4096
CLASS_MAP = {2:0,3:1,4:2,5:3,6:4}
NAMES = ["Ground","Low Veg","Med Veg","High Veg","Building"]; NC=len(NAMES)
from matplotlib.colors import ListedColormap
CMAP = ListedColormap(["#b8a06a","#b6e59e","#5fbf5f","#1f7a1f","#d1483f"])
rng = np.random.default_rng(0); torch.manual_seed(0)
print("device:", dev)

# preprocess: decimated point set with height-above-ground (uses the DTM computed above)
kx,ky,kz,kh,ki,kr,kn,kl = ([] for _ in range(8))
with laspy.open(SRC) as r:
    for pts in r.chunk_iterator(10_000_000):
        c = np.asarray(pts.classification); m = np.isin(c, list(CLASS_MAP))
        m &= rng.random(len(c)) < 0.25
        if not m.any(): continue
        x,y,z = np.asarray(pts.x)[m], np.asarray(pts.y)[m], np.asarray(pts.z)[m]
        col=np.clip(((x-xmin)/RES).astype(np.int64),0,nx-1); row=np.clip(((ymax-y)/RES).astype(np.int64),0,ny-1)
        kx.append(x); ky.append(y); kz.append(z); kh.append(np.clip(z-dtm[row,col],0,None))
        ki.append(np.asarray(pts.intensity)[m].astype(np.float32))
        kr.append(np.asarray(pts.return_number)[m].astype(np.float32))
        kn.append(np.asarray(pts.number_of_returns)[m].astype(np.float32))
        kl.append(np.array([CLASS_MAP[v] for v in c[m]], np.int64))
X=np.concatenate(kx);Y=np.concatenate(ky);Z=np.concatenate(kz);HAG=np.concatenate(kh)
INT=np.concatenate(ki);RR=np.concatenate(kr);NR=np.concatenate(kn);LAB=np.concatenate(kl)
print(f"{len(X):,} training points")

In [ ]:
# block index + spatial split
nbx = int(np.ceil((xmax-xmin)/BLOCK))
bid = (((Y-ymin)/BLOCK).astype(np.int32))*nbx + ((X-xmin)/BLOCK).astype(np.int32)
order = np.argsort(bid,kind="stable"); bids=bid[order]
uniq,starts = np.unique(bids,return_index=True); ends=np.append(starts[1:],len(bids))
blocks = {int(u):order[s:e] for u,s,e in zip(uniq,starts,ends) if e-s>=200}
train_b=[u for u in blocks if xmin+(u%nbx+0.5)*BLOCK < 656000]
val_b  =[u for u in blocks if xmin+(u%nbx+0.5)*BLOCK >= 656000]
tr_idx=np.concatenate([blocks[u] for u in train_b])
INT_m,INT_s = INT[tr_idx].mean(), INT[tr_idx].std()+1e-6
print(f"train blocks={len(train_b)}  val blocks={len(val_b)}")

def make_batch(bs):
    F_,L_=[],[]
    for u in bs:
        idx=blocks[u]; sel=idx[rng.integers(0,len(idx),NPTS)]
        cx=xmin+(u%nbx+0.5)*BLOCK; cy=ymin+(u//nbx+0.5)*BLOCK; zx=Z[sel]
        F_.append(np.stack([(X[sel]-cx)/(BLOCK/2),(Y[sel]-cy)/(BLOCK/2),(zx-zx.min())/20.0,
                  np.clip(HAG[sel]/50,0,1),(INT[sel]-INT_m)/INT_s,RR[sel]/np.maximum(NR[sel],1),
                  NR[sel]/7,(X[sel]-xmin)/2000,(Y[sel]-ymin)/2000],axis=1).astype(np.float32))
        L_.append(LAB[sel])
    return torch.from_numpy(np.stack(F_)), torch.from_numpy(np.stack(L_))

class PointNetSeg(nn.Module):
    def __init__(s,fin,nc):
        super().__init__()
        mlp=lambda i,o: nn.Sequential(nn.Conv1d(i,o,1),nn.BatchNorm1d(o),nn.ReLU())
        s.e1,s.e2,s.e3,s.e4=mlp(fin,64),mlp(64,64),mlp(64,128),mlp(128,1024)
        s.d1,s.d2,s.d3=mlp(1024+64,512),mlp(512,256),mlp(256,128); s.head=nn.Conv1d(128,nc,1)
    def forward(s,x):
        x=x.transpose(1,2); p=s.e2(s.e1(x)); g=s.e4(s.e3(p))
        g=torch.max(g,2,keepdim=True)[0].expand(-1,-1,x.shape[2])
        return s.head(s.d3(s.d2(s.d1(torch.cat([p,g],1))))).transpose(1,2)

In [ ]:
# train (best-checkpoint by val mIoU, gentle class weights, LR decay)
model=PointNetSeg(9,NC).to(dev)
freq=np.array([(LAB[tr_idx]==i).sum() for i in range(NC)],float)
w=torch.tensor(np.clip(np.median(freq)/freq,0.5,3.0),dtype=torch.float32,device=dev)
opt=torch.optim.Adam(model.parameters(),lr=7e-4)
sched=torch.optim.lr_scheduler.StepLR(opt,step_size=5,gamma=0.6)
crit=nn.CrossEntropyLoss(weight=w)
def conf(p,t,nc): return np.bincount(t*nc+p,minlength=nc*nc).reshape(nc,nc)

EPOCHS,BS,PER=15,16,400; best=-1; best_state=None
for ep in range(1,EPOCHS+1):
    model.train(); rng.shuffle(train_b); tot=0; picks=train_b[:PER]
    for i in range(0,len(picks),BS):
        xb,yb=make_batch(picks[i:i+BS]); xb,yb=xb.to(dev),yb.to(dev)
        opt.zero_grad(); loss=crit(model(xb).reshape(-1,NC),yb.reshape(-1))
        loss.backward(); opt.step(); tot+=loss.item()
    sched.step()
    model.eval(); cm=np.zeros((NC,NC),int)
    with torch.no_grad():
        for i in range(0,min(len(val_b),300),BS):
            xb,yb=make_batch(val_b[i:i+BS])
            cm+=conf(model(xb.to(dev)).argmax(-1).cpu().numpy().ravel(),yb.numpy().ravel(),NC)
    oa=np.trace(cm)/cm.sum(); iou=np.diag(cm)/(cm.sum(0)+cm.sum(1)-np.diag(cm)+1e-9); f=""
    if iou.mean()>best: best=float(iou.mean()); best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; f="  <-best"
    print(f"ep{ep:02d} loss={tot/max(1,len(picks)//BS):.3f} valOA={oa:.3f} mIoU={iou.mean():.3f}{f}")
model.load_state_dict(best_state)

In [ ]:
# full validation + confusion matrix + prediction map
model.eval(); cm=np.zeros((NC,NC),int)
with torch.no_grad():
    for i in range(0,len(val_b),BS):
        xb,yb=make_batch(val_b[i:i+BS])
        cm+=conf(model(xb.to(dev)).argmax(-1).cpu().numpy().ravel(),yb.numpy().ravel(),NC)
oa=float(np.trace(cm)/cm.sum()); iou=np.diag(cm)/(cm.sum(0)+cm.sum(1)-np.diag(cm)+1e-9)
metrics=dict(overall_accuracy=round(oa,4),mean_IoU=round(float(iou.mean()),4),
             per_class={NAMES[i]:round(float(iou[i]),4) for i in range(NC)})
json.dump(metrics,open(os.path.join(OUT,"dl_metrics.json"),"w"),indent=2)
torch.save(model.state_dict(),os.path.join(OUT,"dl_pointnet.pt"))
print(json.dumps(metrics,indent=2))

cmn=cm/cm.sum(1,keepdims=True)
fig,ax=plt.subplots(1,2,figsize=(17,7))
ax[0].imshow(cmn,cmap="Blues",vmin=0,vmax=1)
ax[0].set_xticks(range(NC)); ax[0].set_yticks(range(NC))
ax[0].set_xticklabels(NAMES,rotation=45,ha="right"); ax[0].set_yticklabels(NAMES)
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")
for i in range(NC):
    for j in range(NC):
        ax[0].text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",
                   color="white" if cmn[i,j]>.5 else "black",fontsize=8)
ax[0].set_title(f"Confusion  OA={oa:.3f} mIoU={iou.mean():.3f}")
gx,gy,gt,pr=[],[],[],[]
with torch.no_grad():
    for i in range(0,len(val_b),BS):
        bs=val_b[i:i+BS]; xb,yb=make_batch(bs); p=model(xb.to(dev)).argmax(-1).cpu().numpy()
        for j,u in enumerate(bs):
            sel=blocks[u][rng.integers(0,len(blocks[u]),NPTS)]
            gx.append(X[sel]);gy.append(Y[sel]);gt.append(LAB[sel]);pr.append(p[j])
gx=np.concatenate(gx);gy=np.concatenate(gy);pr=np.concatenate(pr)
ax[1].scatter(gx,gy,c=pr,cmap=CMAP,s=.5,vmin=0,vmax=NC-1,linewidths=0)
ax[1].set_title("PointNet prediction (east/val)"); ax[1].set_aspect("equal")
ax[1].set_xlim(656000,xmax); ax[1].set_ylim(ymin,ymax); ax[1].ticklabel_format(style="plain")
plt.tight_layout(); plt.show()

## 10 · Outputs
All results are written to `uiuc_campus_lidar/results/`:

| File | Contents |
|------|----------|
| `dtm.tif`, `chm.tif` | bare-earth terrain & canopy-height rasters (EPSG:6350) |
| `buildings.geojson` | building footprints + `area_m2`, `height_m` (WGS84) |
| `trees.geojson` | tree tops + `height_m`, `crown_m2` (WGS84) |
| `footprint.geojson` | dataset outline for GIS/OSM |
| `dl_metrics.json`, `dl_pointnet.pt` | segmentation metrics & trained weights |

**Extensions:** swap PointNet for PointNet++/KPConv (local context, needs GPU) to sharpen
building edges; add an offset-prediction head for DL *instance* segmentation.